# STEP 1: collect and structure (merge) datasets

### Import libraries 

In [10]:
#Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
#NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Matplotlib is a plotting library for python and pyplot gives us a MatLab like plotting framework. We will use this in our plotter function to plot data.
import matplotlib.pyplot as plt
#Seaborn is a Python data visualization library based on matplotlib. It provides a high-level interface for drawing attractive and informative statistical graphics
import seaborn as sns
from datetime import datetime, timedelta

## Load and view data 

### Indoor temperature

In [11]:
# Get the database from the DataFoundry link
df_indoor = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/Yy9PQlp3clNidmNyc0pYS1JBV1NlQ1JpbGNKWHBIQlVVMjlwQW9nOFY5UT0=", low_memory=False)
df_indoor = df_indoor[(df_indoor.participant == "H1")] # select the correct participant


# # Clean up the dataframe
df_indoor = df_indoor.drop(["device_id", "sender", "participant", "id", "recipient", "pp1", "pp2", "pp3", "activity", "Unnamed: 7"], axis='columns')
df_indoor = df_indoor.rename(columns={"Temperature": "Temperature_indoor"}, errors="raise")
df_indoor = df_indoor.drop(df_indoor[df_indoor.Temperature_indoor < 0].index) ## Drop temperature values under 0
df_indoor = df_indoor.drop(df_indoor[df_indoor.Temperature_indoor > 40].index) ## Drop temperature values over ...
df_indoor['ts'] = pd.to_datetime(df_indoor['ts']) ## Turn timestamp into datetime dtype
df_indoor['ts'] = df_indoor["ts"].dt.round('min')  ##Round the datestamp column to minutes
df_indoor = df_indoor.resample('10min', on='ts').last().reset_index() #get one value per 10 minutes
df_indoor = df_indoor.dropna() ## drop rows with empty (NA) cells
df_indoor = df_indoor.drop_duplicates() ## Drop duplicate rowsindex_list= df_indoor2.Timestamp[(df_indoor2.Timestamp >= "2024-08-08 16:00:00") & (df_indoor2.Timestamp <= "2024-08-08 19:20:00")].index.tolist(

## Drop outliers (e.g. rogue readings, tests, when a sensor is moved)
q_low = df_indoor["Temperature_indoor"].quantile(0.01)
q_hi  = df_indoor["Temperature_indoor"].quantile(0.99)
df_indoor = df_indoor[(df_indoor["Temperature_indoor"] < q_hi) & (df_indoor["Temperature_indoor"] > q_low)]

## Get data after 22 august 18:00 since this is when the sensors were installed
df_indoor = df_indoor[(df_indoor.ts > '2024-08-28 16:00:00')]

display(df_indoor.tail())
display(df_indoor.describe())

,ts,Temperature_indoor
3848,2024-09-18 09:40:00,21.099998
3849,2024-09-18 09:50:00,21.099998
3850,2024-09-18 10:00:00,21.099998
3851,2024-09-18 10:10:00,21.099998
3852,2024-09-18 10:20:00,21.099998


,ts,Temperature_indoor
count,2927,2927.000000
mean,2024-09-07 23:19:53.850358528,22.818961
min,2024-08-28 16:10:00,20.199999
25%,2024-09-02 18:35:00,21.199999
50%,2024-09-07 21:30:00,23.199999
75%,2024-09-12 23:55:00,24.099998
max,2024-09-18 10:20:00,25.800000
std,NaN,1.508891


In [12]:
## Save a copy with sender name specified
df_indoor_save = df_indoor.copy()
df_indoor_save.to_csv(r"C:\Users\20204113\OneDrive - TU Eindhoven\2_Research\2_CoolAI\3_Jupyter_notebooks\H1\DATA (backups)\HH1_indoor1.csv", index = None)

### Window state

In [13]:
# Get the database from the DataFoundry link
df_window = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/QWM2TVQ4OE9KRWF1UU1tRUxzbXJFS1IxM3hhMVZORnNlSlMvRmVyZUFsdz0=", low_memory=False)
df_window = df_window[(df_window.participant == "H1")] # select the correct participant

# Clean up the dataframe
df_window = df_window.drop(["Unnamed: 7", "device_id", "id", "participant", "recipient", "pp1", "pp2", "pp3", "activity", "light", "participant", "curtain"], axis='columns')


# get the left window sensor values (voor = oost)
df_window_voor = df_window.loc[df_window['sender'] == "HH1_window_1"].copy()
df_window_voor["curtain_voor"] = np.where(df_window_voor.loc[:,"distance sensor"] > 35, 1, 0) # set curtain state (0= closed; 1 = open)
df_window_voor = df_window_voor.rename(columns={"distance sensor": "distance_voor"}, errors="raise")
df_window_voor["shade_voor"] = np.where(df_window_voor.loc[:,"light sensor"] < 2500, 1, 0) # set shade (0= no shade; 1 = yes shade)
df_window_voor = df_window_voor.rename(columns={"light sensor": "light_voor"}, errors="raise")
df_window_voor = df_window_voor.drop(["sender", "window", "reed sensor"], axis='columns') # drop window and reed sensor since this sensor does not measure a door state
df_window_voor['ts'] = pd.to_datetime(df_window_voor['ts']) #Turn timestamp into datetime dtype
df_window_voor['ts'] = df_window_voor["ts"].dt.round('min')  ##Round the datestamp column to minutes
df_window_voor = df_window_voor.resample('10min', on='ts').last().reset_index() #get one value per 10 minutes
df_window_voor  = df_window_voor.dropna()
df_window_voor = df_window_voor.drop_duplicates() #Drop duplicate rows


# # get the right window sensor values (achter = west)
df_window_achter = df_window.loc[df_window['sender'] == "HH1_window_2"].copy()
df_window_achter["curtain_achter"] = np.where(df_window_achter.loc[:,"distance sensor"] > 35, 1, 0) # set curtain state (0= closed; 1 = open)
df_window_achter = df_window_achter.rename(columns={"distance sensor": "distance_achter"}, errors="raise")
df_window_achter["shade_achter"] = np.where(df_window_achter.loc[:,"light sensor"] < 2500, 1, 0) # set shade (0= no shade; 1 = yes shade)
df_window_achter = df_window_achter.rename(columns={"light sensor": "light_achter"}, errors="raise")
df_window_achter = df_window_achter.drop(["sender", "window", "reed sensor"], axis='columns') # drop window and reed sensor since this sensor does not measure a door state
df_window_achter['ts'] = pd.to_datetime(df_window_achter['ts']) #Turn timestamp into datetime dtype
df_window_achter['ts'] = df_window_achter["ts"].dt.round('min')  ##Round the datestamp column to minutes
df_window_achter = df_window_achter.resample('10min', on='ts').last().reset_index() #get one value per 10 minutes
df_window_achter  = df_window_achter.dropna()
df_window_achter = df_window_achter.drop_duplicates() #Drop duplicate rows

# # get the right window sensor values
df_windowdoor_voor = df_window.loc[df_window['sender'] == "HH1_windowdoor_3"].copy()
df_windowdoor_voor = df_windowdoor_voor.rename(columns={"reed sensor": "windowdoor_voor"}, errors="raise")
df_windowdoor_voor = df_windowdoor_voor.drop(["sender", "light sensor", "window", "distance sensor"], axis='columns')
df_windowdoor_voor['ts'] = pd.to_datetime(df_windowdoor_voor['ts']) #Turn timestamp into datetime dtype
df_windowdoor_voor['ts'] = df_windowdoor_voor["ts"].dt.round('min')  ##Round the datestamp column to minutes
df_windowdoor_voor = df_windowdoor_voor.resample('10min', on='ts').last().reset_index() #get one value per 10 minutes
df_windowdoor_voor  = df_windowdoor_voor.dropna()
df_windowdoor_voor = df_windowdoor_voor.drop_duplicates() #Drop duplicate rows

# # get the right window sensor values
df_windowdoor_achter = df_window.loc[df_window['sender'] == "HH1_windowdoor_4"].copy()
df_windowdoor_achter = df_windowdoor_achter.rename(columns={"reed sensor": "windowdoor_achter"}, errors="raise")
df_windowdoor_achter = df_windowdoor_achter.drop(["sender", "light sensor", "window", "distance sensor"], axis='columns')
df_windowdoor_achter['ts'] = pd.to_datetime(df_windowdoor_achter['ts']) #Turn timestamp into datetime dtype
df_windowdoor_achter['ts'] = df_windowdoor_achter["ts"].dt.round('min')  ##Round the datestamp column to minutes
df_windowdoor_achter = df_windowdoor_achter.resample('10min', on='ts').last().reset_index() #get one value per 10 minutes
df_windowdoor_achter  = df_windowdoor_achter.dropna()
df_windowdoor_achter = df_windowdoor_achter.drop_duplicates() #Drop duplicate rows

### merge the window dataframes on the timestamps for the last 72 hours
from functools import reduce
# Create a list of DataFrames to merge
dataframes_to_merge = [
    df_window_voor,
    df_window_achter,
    df_windowdoor_voor,
    df_windowdoor_achter
]

# Function to merge two DataFrames
def merge_asof(df1, df2):
    return pd.merge_asof(df1.sort_values('ts'), df2.sort_values('ts'), on='ts', tolerance=pd.Timedelta('120 min'))

# Use reduce to merge all DataFrames in one go
df_window = reduce(merge_asof, dataframes_to_merge)
df_window.tail()
# df_window = df_window.drop_duplicates(subset=['ts']) #Drop duplicate rows

,ts,distance_voor,light_voor,curtain_voor,shade_voor,distance_achter,light_achter,curtain_achter,shade_achter,windowdoor_voor,windowdoor_achter
3324,2024-09-18 09:30:00,67.0,2075.0,1.0,1.0,803.0,3178.0,1.0,0.0,1.0,1.0
3325,2024-09-18 09:40:00,68.0,2037.0,1.0,1.0,803.0,3221.0,1.0,0.0,1.0,1.0
3326,2024-09-18 09:50:00,56.0,2192.0,1.0,1.0,803.0,3255.0,1.0,0.0,1.0,1.0
3327,2024-09-18 10:00:00,56.0,2265.0,1.0,1.0,803.0,3261.0,1.0,0.0,1.0,1.0
3328,2024-09-18 10:10:00,56.0,2254.0,1.0,1.0,803.0,3362.0,1.0,0.0,1.0,1.0


In [14]:
## Save a copy with sender name specified
df_window_save = df_window.copy()
df_window_save.to_csv(r"C:\Users\20204113\OneDrive - TU Eindhoven\2_Research\2_CoolAI\3_Jupyter_notebooks\H1\DATA (backups)\HH1_window.csv", index = None)

### API weather data (outdoor temperature)

In [15]:
# Get the database from the DataFoundry link
df_API = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/N0FsN2loZlVYMWhBaE0rd0l5T2NadzR3YTNBcnovQlJ0SG13dHMxL0U1RT0=")
df_API = df_API[(df_API.participant == "H1")] # select the correct participant

# Clean up the dataframe
df_API = df_API.drop(["id", 'sender', "participant", "device_id", "activity", "pp1", "pp2", "pp3", "Unnamed: 7", "recipient"], axis='columns')
df_API.sender = "HH1_API"
df_API['ts'] = pd.to_datetime(df_API['ts']) ## Turn timestamp into datetime dtype
df_API['Temperature_API'] = np.round(df_API['Temperature_API'] * 10) / 10 ## round temperature to 1 decimal
df_API['Temperature_API_MIN'] = np.round(df_API['Temperature_API_MIN'] * 10) / 10 ## round temperature to 1 decimal
df_API['Temperature_API_MAX'] = np.round(df_API['Temperature_API_MAX'] * 10) / 10 ## round temperature to 1 decimal
df_API = df_API.drop_duplicates() ## Drop duplicate rows

## Drop outliers (e.g. rogue readings, tests, when a sensor is moved)
q_low = df_API["Temperature_API"].quantile(0.01)
q_hi  = df_API["Temperature_API"].quantile(0.99)
df_API = df_API[(df_API["Temperature_API"] < q_hi) & (df_API["Temperature_API"] > q_low)]

## Get data after 22 august 18:00 since this is when the sensors were installed
df_API = df_API[(df_API.ts > '2024-08-22 18:00:00')]

display(df_API.tail())
display(df_API.describe())

,ts,Temperature_API,Temperature_API_MAX,Temperature_API_MIN,weather_description,weather_main
70907,2024-09-18 10:23:09,14.5,15.6,13.8,mist,Mist
70909,2024-09-18 10:24:10,14.5,15.6,13.8,mist,Mist
70911,2024-09-18 10:25:11,14.5,15.6,13.8,mist,Mist
70913,2024-09-18 10:26:13,14.5,15.6,13.8,mist,Mist
70915,2024-09-18 10:27:13,14.5,15.6,13.8,mist,Mist


,ts,Temperature_API,Temperature_API_MAX,Temperature_API_MIN
count,36107,36107.000000,36107.000000,36107.000000
mean,2024-09-05 03:17:34.332982528,17.558465,18.776586,16.586379
min,2024-08-22 18:00:53,5.700000,6.700000,4.500000
25%,2024-08-29 13:57:08.500000,14.400000,15.600000,13.800000
50%,2024-09-05 06:07:44,17.700000,18.900000,16.700000
75%,2024-09-11 16:44:32.500000,20.500000,21.700000,19.400000
max,2024-09-18 10:27:13,29.500000,31.100000,28.800000
std,NaN,4.872400,4.881999,4.804266


## MERGE ON TIMESTAMP

In [16]:
### merge the window dataframes on the timestamps for the last 72 hours
from functools import reduce
# Create a list of DataFrames to merge
dataframes_to_merge = [
    df_indoor,
    df_API,
    df_window,
]

# Function to merge two DataFrames
def merge_asof(df1, df2):
    return pd.merge_asof(df1.sort_values('ts'), df2.sort_values('ts'), on='ts', tolerance=pd.Timedelta('120 min'))

# Use reduce to merge all DataFrames in one go
df_merged = reduce(merge_asof, dataframes_to_merge)
df_merged.tail()

,ts,Temperature_indoor,Temperature_API,Temperature_API_MAX,Temperature_API_MIN,weather_description,weather_main,distance_voor,light_voor,curtain_voor,shade_voor,distance_achter,light_achter,curtain_achter,shade_achter,windowdoor_voor,windowdoor_achter
2922,2024-09-18 09:40:00,21.099998,14.2,15.0,13.7,mist,Mist,68.0,2037.0,1.0,1.0,803.0,3221.0,1.0,0.0,1.0,1.0
2923,2024-09-18 09:50:00,21.099998,14.2,15.5,13.7,mist,Mist,56.0,2192.0,1.0,1.0,803.0,3255.0,1.0,0.0,1.0,1.0
2924,2024-09-18 10:00:00,21.099998,14.2,15.6,13.7,mist,Mist,56.0,2265.0,1.0,1.0,803.0,3261.0,1.0,0.0,1.0,1.0
2925,2024-09-18 10:10:00,21.099998,14.5,15.6,13.7,mist,Mist,56.0,2254.0,1.0,1.0,803.0,3362.0,1.0,0.0,1.0,1.0
2926,2024-09-18 10:20:00,21.099998,14.5,15.6,13.8,mist,Mist,56.0,2254.0,1.0,1.0,803.0,3362.0,1.0,0.0,1.0,1.0


## SAVE AS CSV

In [18]:
df_merged.to_csv(r"C:\Users\20204113\OneDrive - TU Eindhoven\2_Research\2_CoolAI\3_Jupyter_notebooks\H1\DATA (backups)\1_merged.csv", index = None)